In [ ]:
"""5 SEEDS, CELL 1: TRAIN SEEDS 2030 AND 2031 WITH THE ROUND 1 RECIPE, THEN SCORE THEM (G4, about 2 hours 15 minutes).
Same data and settings as seeds 2027, 2028, 2029. Only the random seed differs.
If Colab disconnects: reconnect to a G4 and run THIS cell again. It resumes from the last save (every 100 updates)."""
import sys, gc, json, math, time, uuid, random, shutil, hashlib, subprocess, importlib
from importlib.metadata import version
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch, torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from peft.import_utils import is_torchao_available
from project_utils import save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

DEVICE = "cuda"
MODEL_ID = "google/gemma-4-31B-it"
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
NEW_SEEDS = (2030, 2031)
ROOT = PROJECT / "outputs/runs/step19_five_seeds"
R1_DATA = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
EFFECTIVE_BATCH, LR, SAVE_EVERY, N_TARGET_LAYERS = 16, 1e-4, 100, 230
YES_ID, NO_ID = 10784, 3771
SYSTEM = "You are an expert lexicographer of historical Swedish."

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("0. THE RULE (saved before any training)")
ROOT.mkdir(parents=True, exist_ok=True)
RULE = dict(
    decision_made_in_advance="the average of 5 seeds (2027-2031) REPLACES the average of 3 as system 2, whatever the Swedish number does",
    only_reasons_to_reject=["a new seed fails a health check (all updates logged, loss went down, adapter fingerprint, finite scores, predicts both Yes and No)",
                            "any language (SV, EN, RU, IT, ES, NL) loses more than 0.02 AUC or 0.02 macro-F1 vs the average of 3"],
    cutoff="refitted on the 10 Swedish dev words for the 5-seed average (same grid, ties -> lower cutoff)",
    why="averaging all seeds is decided on principle; adopting it only if it goes up would be picking the lucky result")
if (ROOT / "rule.json").exists():
    assert json.loads((ROOT / "rule.json").read_text()) == RULE, "The saved rule differs from this cell: STOP and paste this output."
    print("Rule already saved earlier and unchanged.")
else:
    save_json(ROOT / "rule.json", RULE)
for k, v in RULE.items():
    print(f"{k:<25}: {v}")

# ---------------------------------------------------------------------------
section("1. CHECK EVERY INPUT FILE BEFORE TOUCHING THE MODEL")
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao is still visible to peft: Runtime > Restart session, then run this cell again."
m1 = json.loads((R1_DATA / "manifest.json").read_text())
train_path = R1_DATA / "pilot_train.jsonl"
assert digest(train_path) == m1["sha256"]["pilot_train.jsonl"], "Round 1 training file changed: STOP."
training = read_rows(train_path)
assert len(training) == 8510
for r in training:
    assert int(r["label"]) in (0, 1) and r["messages"][-1]["role"] == "user"
    assert all(m["role"] in ("system", "user") for m in r["messages"])
enru_manifest = json.loads((ENRU_EVAL / "manifest.json").read_text())
xl_manifest = json.loads((XL_EVAL / "manifest.json").read_text())
eval_sets = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    assert digest(ENRU_EVAL / f"{name}.jsonl") == enru_manifest["output_sha256"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(ENRU_EVAL / f"{name}.jsonl")
for lang in ("IT", "ES", "NL"):
    name = f"{lang}_xlwsd_test"
    assert digest(XL_EVAL / f"{name}.jsonl") == xl_manifest["files"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(XL_EVAL / f"{name}.jsonl")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=False)
eval_sets["SV_dev"] = [dict(id="|".join(u.key), messages=[
    dict(role="system", content=SYSTEM),
    dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                               f"Sense definition (Swedish): {u.definition}\n"
                               f"Sentence from {u.year}: {u.marked_text()}\n\n"
                               "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                               "Answer only Yes or No."))]) for u in sv.usages]
assert len(eval_sets["SV_dev"]) == 1983
TOTAL = math.ceil(len(training) / EFFECTIVE_BATCH)
WARMUP = math.ceil(0.05 * TOTAL)
print(f"Round 1 training file : hash PASS | {len(training):,} examples")
print(f"Evaluation sets       : hash PASS | { {k: len(v) for k, v in eval_sets.items()} }")
print(f"Each seed: {TOTAL} updates, warm-up {WARMUP}, final update used (fixed in advance)")

# ---------------------------------------------------------------------------
section("2. GEMMA 4 31B (bf16)")
if "gemma_model" in globals() or "full_model" in globals() or "BASE" in globals():
    print("Gemma is already loaded in this session: reusing it.")
else:
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 2**30:.0f} GiB | transformers {version('transformers')}")
    assert props.total_memory / 2**30 > 70, "Gemma 31B training needs a GPU with more than 70 GB: use the G4."
    print("Loading Gemma (about 5 minutes on the G4) ...", flush=True)
    gemma_processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
    gemma_tokenizer = gemma_processor.tokenizer
    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
gemma_tokenizer.padding_side = "left"
assert gemma_tokenizer.encode("Yes", add_special_tokens=False) == [YES_ID]
assert gemma_tokenizer.encode("No", add_special_tokens=False) == [NO_ID]

def plain_base():
    """The original Gemma with every adapter removed (unload() removes adapters without merging them)."""
    current = None
    for name in ("full_model", "BASE", "gemma_model"):
        if globals().get(name) is not None:
            current = globals()[name]
            break
    base = current.unload() if isinstance(current, PeftModel) else current
    for name in ("full_model", "gemma_model", "model", "full_optimizer", "params"):
        globals().pop(name, None)
    assert base.config._name_or_path == MODEL_ID and not any("lora_" in n for n, _ in base.named_parameters())
    gc.collect()
    torch.cuda.empty_cache()
    return base

def encode(batch):
    inputs = gemma_processor.apply_chat_template(
        [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False}).to(DEVICE)
    assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
    return inputs

def score_rows(model, rows, destination, tag):
    """Resumable scoring: labels never enter the model; saved to Drive every 100 batches."""
    identity = hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                         ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if destination.exists():
        saved = json.loads(destination.read_text())
        assert saved["input_sha256"] == identity and saved["model_tag"] == tag, f"{destination.name}: saved scores belong to other inputs."
    else:
        saved = dict(input_sha256=identity, model_tag=tag, scores={})
    pending = [r for r in rows if str(r["id"]) not in saved["scores"]]
    model.eval()
    t0 = time.perf_counter()
    try:
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            for b, i in enumerate(range(0, len(pending), 2), 1):
                batch = pending[i:i + 2]
                logits = model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                assert torch.isfinite(logits).all()
                for r, margin in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                    saved["scores"][str(r["id"])] = dict(margin=float(margin))
                del logits
                if b % 100 == 0:
                    save_json(destination, saved)
    finally:
        save_json(destination, saved)
    assert set(saved["scores"]) == {str(r["id"]) for r in rows}
    if pending:
        print(f"      {tag}: {len(rows):,} scores ({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)
    return saved

# ---------------------------------------------------------------------------
def train_seed(seed):
    global BASE, full_model
    run = ROOT / f"seed_{seed}"
    run.mkdir(parents=True, exist_ok=True)
    config = dict(model=MODEL_ID, revision=REVISION, seed=seed, recipe="round 1 (step09) unchanged; only the seed differs",
                  examples=len(training), train_sha256=digest(train_path), micro_batch=1, effective_batch=EFFECTIVE_BATCH,
                  learning_rate=LR, warmup_steps=WARMUP, steps=TOTAL, schedule="linear warmup then linear decay",
                  r=16, alpha=32, dropout=0.05, target="language-model q/k/v/o", base_precision="BF16, no quantization",
                  checkpoint="final update (fixed before training)")
    if (run / "config.json").exists():
        assert json.loads((run / "config.json").read_text()) == json.loads(json.dumps(config)), \
            f"Seed {seed}: settings changed since this run started. STOP and paste this message."
    else:
        save_json(run / "config.json", config)
    if (run / "COMPLETE.json").exists():
        print(f"Seed {seed}: already complete, skipping.")
        return
    BASE = plain_base()
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); random.seed(seed); np.random.seed(seed)
    order = list(training)
    random.Random(seed).shuffle(order)
    targets = [n for n, m in BASE.named_modules() if isinstance(m, torch.nn.Linear) and "language_model" in n.split(".")
               and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")]
    assert len(targets) == N_TARGET_LAYERS, f"Unexpected number of attention layers: {len(targets)}"
    latest_file, final_dir = run / "latest.json", run / "adapter_final"
    state = None
    if (final_dir / "adapter_config.json").exists():
        full_model = PeftModel.from_pretrained(BASE, str(final_dir), is_trainable=False)
        step_done, log = TOTAL, json.loads((run / "training_log.json").read_text())
        print(f"Seed {seed}: training already finished; continuing with scoring.")
    elif latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())["directory"])
        full_model = PeftModel.from_pretrained(BASE, str(latest / "adapter"), is_trainable=True)
        state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)     # our own file
        step_done, log = state["step"], state["log"]
        print(f"Seed {seed}: resuming after update {step_done} of {TOTAL}.")
    else:
        full_model = get_peft_model(BASE, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none"))
        step_done, log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if "lora_B" in n)
        print(f"Seed {seed}: fresh adapter attached (starts from zero).")

    if step_done < TOTAL:
        full_model.config.use_cache = False
        full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        params = [p for p in full_model.parameters() if p.requires_grad]
        assert params and all("lora_" in n for n, p in full_model.named_parameters() if p.requires_grad)
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
        if state:
            optimizer.load_state_dict(state["optimizer"])
            random.setstate(state["python_rng"]); np.random.set_state(state["numpy_rng"]); torch.set_rng_state(state["torch_rng"])
            if state.get("cuda_rng") is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng"])
        del state

        def save_resume(step):
            folder = run / f"resume_{step:04d}_{uuid.uuid4().hex[:8]}"
            folder.mkdir()
            full_model.save_pretrained(folder / "adapter", safe_serialization=True)
            torch.save(dict(step=step, log=log, optimizer=optimizer.state_dict(), python_rng=random.getstate(),
                            numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all()), folder / "training_state.pt")
            previous = json.loads(latest_file.read_text())["directory"] if latest_file.exists() else None
            save_json(latest_file, {"directory": str(folder), "step": step})
            if previous and Path(previous).parent == run and Path(previous).name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            save_json(run / "training_log.json", log)

        full_model.train()
        t0 = time.perf_counter()
        for step in range(step_done + 1, TOTAL + 1):
            batch_rows = order[(step - 1) * EFFECTIVE_BATCH:step * EFFECTIVE_BATCH]
            scale = step / WARMUP if step <= WARMUP else (TOTAL - step + 1) / (TOTAL - WARMUP)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * scale
            optimizer.zero_grad(set_to_none=True)
            loss_sum, correct = 0.0, 0
            for r in batch_rows:
                answer = torch.tensor([YES_ID if int(r["label"]) else NO_ID], device=DEVICE)
                with torch.autocast(DEVICE, dtype=torch.bfloat16):
                    logits = full_model(**encode([r]), use_cache=False, logits_to_keep=1).logits[:, -1, :]
                loss = F.cross_entropy(logits.float(), answer)
                assert torch.isfinite(loss), "Non-finite loss: STOP and paste this message. The last saved checkpoint is intact."
                (loss / len(batch_rows)).backward()
                loss_sum += float(loss.detach())
                correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r["label"])))
                del logits, loss, answer
            norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
            optimizer.step()
            log.append(dict(step=step, loss=loss_sum / len(batch_rows), batch_accuracy=correct / len(batch_rows),
                            grad_norm=norm, lr=LR * scale))
            if step == 1 or step % 50 == 0 or step == TOTAL:
                minutes = (time.perf_counter() - t0) / 60
                print(f"   seed {seed} | update {step:>3}/{TOTAL}: loss {loss_sum / len(batch_rows):.4f} | accuracy {correct / len(batch_rows):.2f} | "
                      f"about {minutes / (step - step_done) * (TOTAL - step):.0f} min of training left", flush=True)
            if step % SAVE_EVERY == 0 and step < TOTAL:
                save_resume(step)
        full_model.save_pretrained(final_dir, safe_serialization=True)
        save_json(run / "training_log.json", log)
        optimizer.zero_grad(set_to_none=True)
        del optimizer, params
        if latest_file.exists():
            previous = Path(json.loads(latest_file.read_text())["directory"])
            if previous.parent == run and previous.name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            latest_file.unlink()
        gc.collect(); torch.cuda.empty_cache()
        losses = [x["loss"] for x in log]
        print(f"Seed {seed}: training done | mean loss first 50 updates {np.mean(losses[:50]):.3f} -> last 50 updates {np.mean(losses[-50:]):.3f}")

    print(f"Seed {seed}: scoring on Swedish dev + the 5 other-language test sets (about 11 minutes)", flush=True)
    (run / "eval_scores").mkdir(exist_ok=True)
    for name, rows_to_score in eval_sets.items():
        score_rows(full_model, rows_to_score, run / "eval_scores" / f"{name}.json", f"seed_{seed}_{name}")
    save_json(run / "COMPLETE.json", dict(seed=seed, adapter=str(final_dir), adapter_sha256=digest(final_dir / "adapter_model.safetensors"),
                                          eval_scores={name: f"eval_scores/{name}.json" for name in eval_sets}))
    print(f"Seed {seed}: COMPLETE")
    BASE = plain_base()

for i, seed in enumerate(NEW_SEEDS, 3):
    section(f"{i}. SEED {seed}: TRAIN, THEN SCORE (about 1 hour)")
    train_seed(seed)

section("SUMMARY")
for seed in NEW_SEEDS:
    print(f"Seed {seed}: {'COMPLETE' if (ROOT / f'seed_{seed}' / 'COMPLETE.json').exists() else 'NOT complete'}")
print(f"Saved in: {ROOT}")
print("System 2 (3 seeds) is untouched until the decision cell runs.")
print("Next: paste this output, then run the decision cell (no GPU needed).")

In [ ]:
"""5 SEEDS, CELL 2: THE DECISION + FREEZE (no GPU work, about 3 minutes).
Applies the rule saved before training. If it passes, freezes the 5-seed average as the main system."""
import sys, json, hashlib, subprocess, importlib.util
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from sklearn.metrics import roc_auc_score
from project_utils import save_json, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

ROOT = PROJECT / "outputs/runs/step19_five_seeds"
ENS_ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
PRIMARY = PROJECT / "outputs/frozen/primary_gemma4_31b_lora_macro.json"
FREEZE = PROJECT / "outputs/frozen/main_system_ensemble5_gemma4_31b_lora.json"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
XL_SCORES = PROJECT / "outputs/runs/step11_xlwsd_scores/adapter_6f3d236ac50e"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
EXTERNAL = ("EN_wic_dev", "RU_axolotl_test_derived", "IT_xlwsd_test", "ES_xlwsd_test", "NL_xlwsd_test")
SEEDS, NEW = (2027, 2028, 2029, 2030, 2031), (2030, 2031)
E3, E5 = "average of 3", "average of 5"

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for row in report["per_word"]:
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        tn = row["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        row["positive_f1"], row["f1"] = row["f1"], sum(parts) / len(parts)
        languages[row["language"]].append(row["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

def pooled_macro(y, s, t):
    y, p = np.asarray(y), (np.asarray(s) >= t).astype(int)
    f = lambda c: 2 * np.sum((y == c) & (p == c)) / max(np.sum(y == c) + np.sum(p == c), 1)
    return float((f(1) + f(0)) / 2)

def seed_dir(seed):
    return ENS_ROOT / f"seed_{seed}" if seed in (2028, 2029) else ROOT / f"seed_{seed}"

# ---------------------------------------------------------------------------
section("1. THE RULE AND THE FIVE ADAPTERS")
RULE = json.loads((ROOT / "rule.json").read_text())
for k, v in RULE.items():
    print(f"{k:<25}: {v}")
primary = json.loads(PRIMARY.read_text())
adapters = [dict(seed=2027, adapter_dir=primary["adapter_dir"], sha256=primary["adapter_sha256"])]
for seed in SEEDS[1:]:
    p = seed_dir(seed) / "COMPLETE.json"
    assert p.exists(), f"Seed {seed} is not complete: STOP."
    done = json.loads(p.read_text())
    adapters.append(dict(seed=seed, adapter_dir=done["adapter"], sha256=done["adapter_sha256"]))
print()
for a in adapters:
    ok = digest(Path(a["adapter_dir"]) / "adapter_model.safetensors") == a["sha256"]
    assert ok, f"Seed {a['seed']}: adapter file changed: STOP and paste this output."
    print(f"Seed {a['seed']}: fingerprint PASS ({a['sha256'][:16]}...)")
assert len({a["sha256"] for a in adapters}) == 5, "Two adapters are identical: STOP."
print("All 5 adapters are different files: PASS")

# ---------------------------------------------------------------------------
section("2. LOAD SWEDISH SCORES")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
keys = sorted(gold)
def by_key(saved, tag):
    assert saved["model_tag"] == tag, f"Unexpected scores: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
single = {2027: by_key(json.loads((R1_RUN / "SV_scores.json").read_text()), "chosen_532_SV")}
for seed in SEEDS[1:]:
    single[seed] = by_key(json.loads((seed_dir(seed) / "eval_scores/SV_dev.json").read_text()), f"seed_{seed}_SV_dev")
for seed, s in single.items():
    v = np.array(list(s.values()))
    assert np.isfinite(v).all() and np.abs(v).max() < 100, f"Seed {seed}: scores outside the grid: STOP."
ens3 = {k: float(np.mean([single[s][k] for s in SEEDS[:3]])) for k in keys}
ens5 = {k: float(np.mean([single[s][k] for s in SEEDS])) for k in keys}
print(f"{len(keys):,} Swedish usages | 5 single models + average of 3 + average of 5")

# ---------------------------------------------------------------------------
if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text()
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")
def nested(s, name):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(s), GRID)])
    finally:
        cv.score = positive_score
def fitted(s):
    v = np.array([s[k] for k in keys])
    t = max(GRID, key=lambda x: (macro_score(gold, dict(zip(keys, (v >= x).astype(int).tolist())))["average"], -x))
    return float(t), macro_score(gold, dict(zip(keys, (v >= t).astype(int).tolist())))["average"]

section("3. SWEDISH DEV, OFFICIAL METRIC (nested leave-one-word-out)")
res = {f"seed {s}": nested(single[s], f"s{s}") for s in SEEDS}
res[E3], res[E5] = nested(ens3, "e3"), nested(ens5, "e5")
for name in res:
    assert abs(official_check(res[name]["predictions"]) - res[name]["report"]["average"]) < 1e-12, f"{name}: official scorer disagrees: STOP."
ENS3_SAVED = json.loads((ENS_ROOT / "ensemble_decision.json").read_text())["swedish"]["ensemble (average of 3)"]
assert abs(res[E3]["report"]["average"] - ENS3_SAVED) < 1e-9, "Average of 3 no longer reproduces 0.7664: STOP and paste this output."
print(f"Average of 3 reproduces {ENS3_SAVED:.4f}: PASS | official scorer agrees on all 7: PASS\n")
singles = [res[f"seed {s}"]["report"]["average"] for s in SEEDS]
for s in SEEDS:
    print(f"   single model, seed {s}   macro-F1 {res[f'seed {s}']['report']['average']:.4f}")
print(f"   ----------------------------------------")
print(f"   single model: mean {np.mean(singles):.4f}, spread (std) {np.std(singles, ddof=1):.4f}, range {min(singles):.4f} to {max(singles):.4f}")
print(f"   {E3:<27} macro-F1 {res[E3]['report']['average']:.4f}")
print(f"   {E5:<27} macro-F1 {res[E5]['report']['average']:.4f}")
per = {n: {row["word"]: row["f1"] for row in res[n]["report"]["per_word"]} for n in (E3, E5)}
print(f"\n{'word':<11}{E3:>14}{E5:>14}{'change':>9}")
for w in sorted(per[E3]):
    print(f"{w:<11}{per[E3][w]:>14.3f}{per[E5][w]:>14.3f}{per[E5][w] - per[E3][w]:>+9.3f}")
paired = word_bootstrap(res[E5]["report"], other=res[E3]["report"])
print(f"\nAverage of 5 minus average of 3: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f})")
cut3, cut5 = fitted(ens3), fitted(ens5)
print(f"Cutoffs fitted on all 10 words: average of 3 {cut3[0]:+.2f} | average of 5 {cut5[0]:+.2f} (in-sample {cut5[1]:.4f})")
auc_sv = {E3: float(roc_auc_score([gold[k] for k in keys], [ens3[k] for k in keys])),
          E5: float(roc_auc_score([gold[k] for k in keys], [ens5[k] for k in keys]))}

# ---------------------------------------------------------------------------
section("4. HEALTH CHECKS FOR THE TWO NEW SEEDS")
health = {}
for seed in NEW:
    run = seed_dir(seed)
    log = json.loads((run / "training_log.json").read_text())
    steps = json.loads((run / "config.json").read_text())["steps"]
    losses = [x["loss"] for x in log]
    first, last = float(np.mean(losses[:50])), float(np.mean(losses[-50:]))
    t = fitted(single[seed])[0]
    yes = sum(single[seed][k] >= t for k in keys)
    health[seed] = {f"all {steps} updates in the log": len(log) == steps,
                    f"loss went down ({first:.3f} -> {last:.3f})": last < first,
                    f"predicts both Yes and No ({yes:,} Yes of {len(keys):,})": 0 < yes < len(keys)}
    print(f"Seed {seed}:")
    for name, ok in health[seed].items():
        print(f"   {'PASS' if ok else 'FAIL'}  {name}")
health_ok = all(all(h.values()) for h in health.values())

# ---------------------------------------------------------------------------
section("5. EVERY LANGUAGE: average of 5 vs average of 3 (AUC and macro-F1, each with its own Swedish cutoff)")
langs = {"SV_dev (nested)": dict(auc={E3: auc_sv[E3], E5: auc_sv[E5]},
                                 macro_f1={E3: res[E3]["report"]["average"], E5: res[E5]["report"]["average"]})}
for name in EXTERNAL:
    rows = read_jsonl((ENRU_EVAL if name[:2] in ("EN", "RU") else XL_EVAL) / f"{name}.jsonl")
    ids, y = [str(row["id"]) for row in rows], [int(row["label"]) for row in rows]
    if name[:2] in ("EN", "RU"):
        r1 = json.loads((R1_RUN / f"{name}_scores.json").read_text())["scores"]
        per_seed = [[r1[i]["margin"] for i in ids]]
    else:
        r1 = json.loads((XL_SCORES / f"{name[:2]}_trained_scores.json").read_text())
        per_seed = [[r1[i] for i in ids]]
    for seed in SEEDS[1:]:
        saved = json.loads((seed_dir(seed) / f"eval_scores/{name}.json").read_text())
        assert saved["model_tag"] == f"seed_{seed}_{name}", f"Unexpected scores: {saved['model_tag']}"
        per_seed.append([saved["scores"][i]["margin"] for i in ids])
    per_seed = np.array(per_seed)
    assert np.isfinite(per_seed).all(), f"{name}: non-finite scores: STOP."
    s3, s5 = per_seed[:3].mean(axis=0), per_seed.mean(axis=0)
    langs[name] = dict(auc={E3: float(roc_auc_score(y, s3)), E5: float(roc_auc_score(y, s5))},
                       macro_f1={E3: pooled_macro(y, s3, cut3[0]), E5: pooled_macro(y, s5, cut5[0])})
all_ok = True
print(f"{'data':<26}{'AUC 3':>8}{'AUC 5':>8}{'change':>8}   {'mF1 3':>8}{'mF1 5':>8}{'change':>8}  rule")
for name, d in langs.items():
    da, df = d["auc"][E5] - d["auc"][E3], d["macro_f1"][E5] - d["macro_f1"][E3]
    ok = bool(da >= -0.02 and df >= -0.02)
    d["rule_pass"] = ok
    all_ok &= ok
    print(f"{name:<26}{d['auc'][E3]:>8.4f}{d['auc'][E5]:>8.4f}{da:>+8.4f}   {d['macro_f1'][E3]:>8.4f}{d['macro_f1'][E5]:>8.4f}{df:>+8.4f}  {'PASS' if ok else 'FAIL'}")
print("External sets are not competition data.")

# ---------------------------------------------------------------------------
section("6. THE RULE (saved before training)")
print(f"New seeds pass every health check               : {'PASS' if health_ok else 'FAIL'}")
print(f"No language loses > 0.02 AUC or macro-F1        : {'PASS' if all_ok else 'FAIL'}")
adopt = bool(health_ok and all_ok)
save_json(ROOT / "five_seed_decision.json", dict(
    metric="per-word macro-F1, official scorer commit 0e69238, nested leave-one-word-out",
    swedish={n: res[n]["report"]["average"] for n in res},
    single_model_summary=dict(mean=float(np.mean(singles)), std=float(np.std(singles, ddof=1)), min=min(singles), max=max(singles)),
    per_word=per, paired_5_minus_3=paired, cutoffs={E3: cut3[0], E5: cut5[0]}, in_sample_5=cut5[1],
    health={str(s): {k: bool(v) for k, v in h.items()} for s, h in health.items()}, languages=langs,
    rule=RULE, adopt=adopt))

section("7. FREEZE")
if adopt:
    assert not FREEZE.exists(), "The 5-seed main-system file already exists: STOP, do not overwrite."
    save_json(FREEZE, dict(
        status="MAIN SYSTEM. Average of 5 Gemma 4 31B LoRA adapters (seeds 2027-2031). Backup: system 1 (seed 2027 alone).",
        frozen_utc=datetime.now(timezone.utc).isoformat(),
        model_id=primary["model_id"], revision=primary["revision"], adapters=adapters,
        combine="score = mean over the 5 adapters of logit(Yes) - logit(No); predict 1 if score >= final_threshold",
        final_threshold=cut5[0],
        threshold_rule="single cutoff fitted on all 10 Swedish dev words for macro-F1, predeclared grid, ties -> lower cutoff",
        prompt=primary["prompt"], scoring=primary["scoring"],
        training="each adapter: round 1 recipe and data (8,510 RU+EN examples), final update; only the random seed differs",
        metric="per-word macro-F1 (official scorer commit 0e69238)",
        swedish_nested_cv_macro_f1=res[E5]["report"]["average"], swedish_in_sample_macro_f1=cut5[1],
        single_model_summary=dict(mean=float(np.mean(singles)), std=float(np.std(singles, ddof=1))),
        languages=langs, decision_file=str(ROOT / "five_seed_decision.json"), system1_config=str(PRIMARY.relative_to(PROJECT))))
    print(f"FROZEN as the main system: {FREEZE.name}")
    print(f"Swedish {res[E5]['report']['average']:.4f} (nested) | cutoff {cut5[0]:+.2f} | single models {np.mean(singles):.4f} +/- {np.std(singles, ddof=1):.4f}")
else:
    print("NOT ADOPTED: something broke (see sections 4 and 5). The average of 3 stays the main system. Nothing frozen.")

section("SUMMARY")
print(f"Single model (5 seeds) : {np.mean(singles):.4f} +/- {np.std(singles, ddof=1):.4f}")
print(f"Average of 3           : {res[E3]['report']['average']:.4f}")
print(f"Average of 5           : {res[E5]['report']['average']:.4f}   <- {'MAIN SYSTEM' if adopt else 'not adopted'}")
print("Paste this whole output.")